In [317]:
import matplotlib.pyplot as plt
import seaborn as sns

In [318]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "online_shoppers_intention.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "henrysue/online-shoppers-intention",
  file_path
)

In [319]:
# Eliminación de duplicados
df_clean = df.drop_duplicates(keep='first')

In [320]:
n = df_clean.shape[0]
d = df_clean.shape[1]

print(f'Conjunto de datos con {n} filas y {d} columnas')

Conjunto de datos con 12205 filas y 18 columnas


In [321]:
print('[Columnas]')
print(df_clean.columns)

[Columnas]
Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='str')


In [322]:
import pandas as pd

meses_map = {'Jan': 1, 'Feb':2, 'Mar':3, 'Apr': 4, 'May':5, 'June':6, 'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}
df_clean['Month'] = df_clean['Month'].map(meses_map)
df_clean = pd.get_dummies(df_clean, columns=['VisitorType', 'OperatingSystems', 'Browser', 'Region', 'TrafficType'], drop_first=True, dtype=int)

df_clean['Weekend'] = df_clean['Weekend'].astype(int)
df_clean['Revenue'] = df_clean['Revenue'].astype(int)

In [323]:
df_clean.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,TrafficType_11,TrafficType_12,TrafficType_13,TrafficType_14,TrafficType_15,TrafficType_16,TrafficType_17,TrafficType_18,TrafficType_19,TrafficType_20
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [324]:
n = df_clean.shape[0]
d = df_clean.shape[1]

print(f'Conjunto de datos con {n} filas y {d} columnas')

Conjunto de datos con 12205 filas y 61 columnas


In [325]:
y = df_clean['Revenue']
X = df_clean.drop('Revenue', axis=1)

### Código que lleve los datos a 2D

In [326]:
# --- TODO ---

In [327]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [328]:
from collections import Counter

print('Distribución para y_train:',Counter(y_train))

Distribución para y_train: Counter({0: 8238, 1: 1526})


In [329]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print('Distribución para y_train re-sampleado:', Counter(y_train_smote))

Distribución para y_train re-sampleado: Counter({1: 8238, 0: 8238})


In [330]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(random_state=42)
lr.fit(X_train_smote, y_train_smote)

proba = lr.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.94      0.89      0.92      2059
           1       0.54      0.71      0.62       382

    accuracy                           0.86      2441
   macro avg       0.74      0.80      0.77      2441
weighted avg       0.88      0.86      0.87      2441



/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### RandomForest - SIN SMOTE

In [331]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# 1. Definimos el modelo base
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# 2. Definimos la "rejilla" de parámetros a probar
param_grid = {
    'n_estimators': [100, 200, 300],        # Número de árboles
    'max_depth': [None, 10, 20],            # Profundidad (evita overfitting)
    'min_samples_split': [2, 5, 10],        # Mínimo de muestras para dividir un nodo
    'max_features': ['sqrt', 'log2']        # Número de variables a considerar en cada división
}

# 3. Configuramos la búsqueda
# Usamos scoring='f1' porque es lo que queremos optimizar para la clase minoritaria
grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5,                 # Validación cruzada de 5 carpetas
    scoring='f1',         # ¡CLAVE! Optimiza para F1, no para Accuracy
    n_jobs=-1,            # Usa todos los núcleos de tu procesador
    verbose=1
)

grid_search.fit(X_train, y_train)

# 5. Resultados
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor F1-Score en entrenamiento: {grid_search.best_score_}")

# 6. Evaluar en el set de TEST real
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Mejores parámetros: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 300}
Mejor F1-Score en entrenamiento: 0.6750502301952142
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      2059
           1       0.65      0.73      0.68       382

    accuracy                           0.90      2441
   macro avg       0.80      0.83      0.81      2441
weighted avg       0.90      0.90      0.90      2441



### RandomForest - CON SMOTE

In [332]:
from sklearn.model_selection import GridSearchCV

grid_search2 = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5,                 
    scoring='f1',         
    n_jobs=-1,           
    verbose=1
)

grid_search2.fit(X_train_smote, y_train_smote)

print(f"Mejores parámetros: {grid_search2.best_params_}")
print(f"Mejor F1-Score en entrenamiento: {grid_search2.best_score_}")

best_rf = grid_search2.best_estimator_
y_pred2 = best_rf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred2))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Mejores parámetros: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Mejor F1-Score en entrenamiento: 0.9010861065656612
              precision    recall  f1-score   support

           0       0.94      0.92      0.93      2059
           1       0.62      0.71      0.66       382

    accuracy                           0.89      2441
   macro avg       0.78      0.81      0.80      2441
weighted avg       0.89      0.89      0.89      2441



### XGBoost - SIN SMOTE

In [333]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100, 200],           # Número de árboles
    'max_depth': [3, 6, 9],               # Profundidad (XGBoost prefiere árboles más pequeños que RF)
    'learning_rate': [0.01, 0.1, 0.2],    # Qué tan rápido aprende (paso de gradiente)
    'subsample': [0.8, 1.0],              # % de datos usados para cada árbol (evita overfitting)
    'colsample_bytree': [0.8, 1.0]        # % de variables usadas para cada árbol
}

# 3. Configuramos la búsqueda enfocada en F1
grid_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=5,                 
    scoring='f1',        
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

print(f"Mejores parámetros XGB: {grid_xgb.best_params_}")
print(f"Mejor F1-Score en entrenamiento: {grid_xgb.best_score_}")

best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_xgb))

Fitting 5 folds for each of 72 candidates, totalling 360 fits


/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are no

Mejores parámetros XGB: {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Mejor F1-Score en entrenamiento: 0.6617524925409487
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      2059
           1       0.71      0.62      0.67       382

    accuracy                           0.90      2441
   macro avg       0.82      0.79      0.80      2441
weighted avg       0.90      0.90      0.90      2441



/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### XGBoost - CON SMOTE

In [334]:
from sklearn.model_selection import GridSearchCV

grid_xgb2 = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=5,                 
    scoring='f1',        
    n_jobs=-1,
    verbose=1
)

grid_xgb2.fit(X_train_smote, y_train_smote)

print(f"Mejores parámetros XGB: {grid_xgb2.best_params_}")
print(f"Mejor F1-Score en entrenamiento: {grid_xgb2.best_score_}")

best_xgb = grid_xgb2.best_estimator_
y_pred_xgb2 = best_xgb.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_xgb2))

/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fitting 5 folds for each of 72 candidates, totalling 360 fits


/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/alvaro/Desktop/IA/4º/2ºcuatrimestre/Aplicaciones de la IA/Proyetos/proyecto-1-AIA-Marketing_Digital/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:00:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are no

Mejores parámetros XGB: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 9, 'n_estimators': 200, 'subsample': 0.8}
Mejor F1-Score en entrenamiento: 0.8996626091805131
              precision    recall  f1-score   support

           0       0.95      0.91      0.93      2059
           1       0.60      0.74      0.66       382

    accuracy                           0.88      2441
   macro avg       0.77      0.82      0.79      2441
weighted avg       0.89      0.88      0.89      2441



"Aunque un F1-Score de 0.68 pueda parecer alejado de la perfección, en el contexto de la navegación web representa un salto competitivo inmenso. El modelo permite pasar de una estrategia masiva e ineficiente a una micro-segmentada, donde 2 de cada 3 impactos publicitarios se realizan sobre usuarios con una intención real de compra, optimizando el retorno de inversión (ROI) y reduciendo el desperdicio de presupuesto en un 400% respecto a una campaña sin segmentación."